## Extracción de imágenes de los Reportes de CENAPRED mediante Web Scraping

## Instalar e importar librerías necesarias

In [2]:
import requests
from bs4 import BeautifulSoup
import os

In [3]:
# Página de la fecha más actual

url = "https://www.gob.mx/cenapred/es/archivo/articulos?\
filter_origin=archive&idiom=es&order=DESC&page=1"

response = requests.get(url)

page_content = response.content

In [4]:
url

'https://www.gob.mx/cenapred/es/archivo/articulos?filter_origin=archive&idiom=es&order=DESC&page=1'

In [5]:
# Revisar que la solicitud htpp haya sido respondida
response.status_code

200

In [7]:
soup = BeautifulSoup(page_content, 'html5lib')

In [11]:
image_tags = soup.find_all('img')
print(len(image_tags))
#image_tags
image_tags[0]

9


<img 600="" alt='\"Se' class='\"img_curve\"' de="" detectaron 11 exhalaciones y="" minutos="" onerror="\&quot;this.src='https://framework-gb.cdn.gob.mx/main/missing-photo.jpg';\&quot;" src='\"https://www.gob.mx/cms/uploads/article/main_image/171343/cover_1.jpg\"' tremor.\"="" width='\"350\"'/>

In [12]:
for i, image in enumerate(image_tags, start=1):
    print(image.get('src'))

\"https://www.gob.mx/cms/uploads/article/main_image/171343/cover_1.jpg\"
\"https://www.gob.mx/cms/uploads/article/main_image/171320/cover_06_54_20.jpg\"
\"https://www.gob.mx/cms/uploads/article/main_image/171289/cover_7.jpg\"
\"https://www.gob.mx/cms/uploads/article/main_image/171223/cover_2.jpg\"
\"https://www.gob.mx/cms/uploads/article/main_image/171213/cover_blog.jpg\"
\"https://www.gob.mx/cms/uploads/article/main_image/171204/cover_blog.jpg\"
\"https://www.gob.mx/cms/uploads/article/main_image/171140/cover_000682390W.jpg\"
\"https://www.gob.mx/cms/uploads/article/main_image/171159/cover_webcams_altzomoni.jpg\"
\"https://www.gob.mx/cms/uploads/article/main_image/171107/cover_WhatsApp_Image_2026-11-05_at_10.01.28.jpeg\"


In [38]:
image_urls = []
for image_tag in image_tags:
    image_urls.append(image_tag['src'])
image_urls

['\\"https://www.gob.mx/cms/uploads/article/main_image/171289/cover_7.jpg\\"',
 '\\"https://www.gob.mx/cms/uploads/article/main_image/171223/cover_2.jpg\\"',
 '\\"https://www.gob.mx/cms/uploads/article/main_image/171213/cover_blog.jpg\\"',
 '\\"https://www.gob.mx/cms/uploads/article/main_image/171204/cover_blog.jpg\\"',
 '\\"https://www.gob.mx/cms/uploads/article/main_image/171140/cover_000682390W.jpg\\"',
 '\\"https://www.gob.mx/cms/uploads/article/main_image/171159/cover_webcams_altzomoni.jpg\\"',
 '\\"https://www.gob.mx/cms/uploads/article/main_image/171107/cover_WhatsApp_Image_2026-11-05_at_10.01.28.jpeg\\"',
 '\\"https://www.gob.mx/cms/uploads/article/main_image/171041/cover_D_a_del_Tsunami.jpg\\"',
 '\\"https://www.gob.mx/cms/uploads/article/main_image/171069/cover_07_53_07.jpg\\"']

In [13]:
def clean_image_url(url):
    return url.replace('\\', '').replace('"', '')

image_urls = []

for page in range(1, 31):
    url = f"https://www.gob.mx/cenapred/es/archivo/articulos?filter_origin=archive&idiom=es&order=DESC&page={page}"
    response = requests.get(url)
    page_content = response.content
    
    soup = BeautifulSoup(page_content, 'html5lib')
    image_tags = soup.find_all('img')
    
    for image_tag in image_tags:
        raw_url = image_tag.get('src')
        if raw_url:
            cleaned_url = clean_image_url(raw_url)
            image_urls.append(cleaned_url)
image_urls[:5]

['https://www.gob.mx/cms/uploads/article/main_image/171343/cover_1.jpg',
 'https://www.gob.mx/cms/uploads/article/main_image/171320/cover_06_54_20.jpg',
 'https://www.gob.mx/cms/uploads/article/main_image/171289/cover_7.jpg',
 'https://www.gob.mx/cms/uploads/article/main_image/171223/cover_2.jpg',
 'https://www.gob.mx/cms/uploads/article/main_image/171213/cover_blog.jpg']

In [14]:
# Crear carpeta para guardar las imágenes si no existe
output_folder = 'images_2024'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Descargar las imágenes
for i, img_url in enumerate(image_urls):
    try:
        # Descargar la imagen
        img_response = requests.get(img_url)
        img_response.raise_for_status()  # Verificar que la solicitud fue exitosa
        
        # Definir el nombre del archivo
        img_name = os.path.join(output_folder, f'image_{i+1}.jpg')
        
        # Guardar la imagen en el archivo
        with open(img_name, 'wb') as f:
            f.write(img_response.content)
        
        print(f'Descargada imagen {i+1}: {img_url}')
    except Exception as e:
        print(f'Error al descargar la imagen {i+1}: {e}')

print(f'Total de imágenes descargadas: {len(image_urls)}')

Descargada imagen 1: https://www.gob.mx/cms/uploads/article/main_image/171343/cover_1.jpg
Descargada imagen 2: https://www.gob.mx/cms/uploads/article/main_image/171320/cover_06_54_20.jpg
Descargada imagen 3: https://www.gob.mx/cms/uploads/article/main_image/171289/cover_7.jpg
Descargada imagen 4: https://www.gob.mx/cms/uploads/article/main_image/171223/cover_2.jpg
Descargada imagen 5: https://www.gob.mx/cms/uploads/article/main_image/171213/cover_blog.jpg
Descargada imagen 6: https://www.gob.mx/cms/uploads/article/main_image/171204/cover_blog.jpg
Descargada imagen 7: https://www.gob.mx/cms/uploads/article/main_image/171140/cover_000682390W.jpg
Descargada imagen 8: https://www.gob.mx/cms/uploads/article/main_image/171159/cover_webcams_altzomoni.jpg
Descargada imagen 9: https://www.gob.mx/cms/uploads/article/main_image/171107/cover_WhatsApp_Image_2026-11-05_at_10.01.28.jpeg
Descargada imagen 10: https://www.gob.mx/cms/uploads/article/main_image/171041/cover_D_a_del_Tsunami.jpg
Descargada